# The Fenna–Matthews–Olson (FMO) complex


The FMO complex is a a water-soluble complex found in green sulfur bacteria. It plays a key role in photosynthesis by mediating energy transfer between light-absorbing components, and this process can be modeled as an open quantum system.

The FMO complex is modeled as a network of sites (usually 7 or 8), each representing a bacteriochlorophyll molecule. The system Hamiltonian is written as:

$$
H_S=\sum_n\epsilon_n \ket{n}\bra{n}+ \sum_{n\ne m}J_{nm}\ket{n}\bra{m}
$$

where $\epsilon_n$ are site energies and $J_{nm}$ are electronic couplings between sites.

From the open system perspective, the FMO model provides a canonical example of excitation energy transfer in a noisy, dissipative environment allowing us to study. For the system-bath interaction, each FMO site $n$ is coupled to a local bosonic bath representing protein and solvent motions:

$$
H_{SB}=\sum_n \ket{n}\bra{n}\otimes \sum_jg_{nj}(a_{nj}+a_{nj}^\dagger)
$$

Here are some key points in understanding the interacting Hamiltonian:
- Each site has its own local environment, and that environment only interacts with the system when the excitation is at that site.
- There are no transitions between different sites induced by the bath — only energy fluctuations (i.e. dephasing).
- Energy of each site is shifted because of the bath, which is similar to the “dressed state”.

# Setup

In [1]:
from math import ceil
import os
import json as json
import numpy as np
from tqdm import tqdm

from tenso.prototypes.heom import system_multibath
from tenso.prototypes.bath import gen_bcf
import matplotlib.pyplot as plt

# Bath Correlation Function (BCF)

In [2]:
bath = gen_bcf(
        re_d=[35.0], #Here we only consider the Drude-Lorentz Bath
        width_d=[106.17674918],
        re_b=[26.24, 13.832, 101.479],
        freq_b=[160, 247, 763],
        width_b=[133, 53, 76],
        temperature=300,
        decomposition_method='Pade',
        n_ltc=1,
    )

# The Hamiltonian of the FMO complex and the initial state

In [3]:
h = np.array([[310, -98, 6],
              [-98, 230, 30],
              [6, 30, 0]],
                 dtype=np.complex128) #This is the system Hamiltonian of the FMO complex
    
avg = np.diag(h).mean()
h -= avg * np.eye(3) # Shifting the Hamiltonian’s diagonal by subtracting the average energy, convenient for computation


sys_ops = []
end_time = 1000.0
dt = 1
for i in range(3):
    op_i = np.zeros((3, 3))
    op_i[i, i] = 1.0
    sys_ops.append(op_i) # The system operator of a site is a list of projective operators [|n><n|]

wfn = np.zeros(3)
wfn[0] = 1.0 #The initial state is a site state in site_1

# System-Multibath Propagator

**TENSO** provides the function `system_multibath()` to simulate open quantum systems in which different sites are coupled to distinct baths. An example of such system is the FMO model described above. This function gathers all parameters related to the system, the propagation scheme, and the influence of the baths on the system dynamics.

- `fname`: Specifies the output file name. In this example, it is `"3_level_FMO"`, corresponding to the `out` variable.
- `init_rdo`: Represents the outer product between $|\Psi\rangle$ and $\langle \Psi|$. Here, $|\Psi\rangle$ corresponds to the `wfn` variable. This argument defines the initial state of the system. In this example, the initial state is the site state in site_1.
- `sys_ham`: The system Hamiltonian. In this example, we shift the Hamiltonian’s diagonal elements by subtracting the average energy, which is for computation purpose. 
- `sys_op`: The system operator ($Q_S$) entering the system-bath interaction Hamiltonian ($H_{SB}$). Each site couples to its own bath, so each site-bath coupling has its own system operator. `sys_op` should be a list of system operators where each system operator is associated to a site and a bath. In this example, each element is the projective operator on the site state, expressed as $\ket{n}\bra{n}$. 
- `bath_correlation`: The bath correlation function (BCF) is a list of `bath` variables. In the list, each element is a seperated bath created by `get_bcf()`, coupled to the corresponding site through the interaction Hamiltonian.
- `dim`: The hierarchy depth. Higher values of `dim` typically yield more accurate results.
- `end_time`: Final time of the simulation.
- `step_timp`: Time step (`dt`) used for propagation.

`**tqdm**` is employed to display the progress bar during the simulation runtime.

In [ ]:

out = "3_level_FMO_structured"
propagator = system_multibath( 
        fname=out,
        init_rdo=np.outer(wfn, wfn.conj()),
        sys_ham=h,
        sys_ops=sys_ops,
        bath_correlations=[bath] * 3, #Each site has its own bath, different baths are seperated and independent
        end_time=end_time,
        step_time=dt,
        dim=10
    )

progress_bar = tqdm(propagator, total=ceil(end_time / dt)) #Show the progress bar
for _t in (progress_bar):
    progress_bar.set_description(f'@{_t:.2f} fs')

  0%|          | 0/1000 [00:00<?, ?it/s]

{'auxiliary_ps_method': 'ps2', 'auxiliary_step_time': None, 'cache_svd_info': True, 'dim': 10, 'dvr_length': 32, 'dvr_type': 'sine', 'end_time': 1000.0, 'frame_method': 'tree2', 'load_checkpoint_from_file': False, 'max_auxiliary_rank': 64, 'max_auxiliary_steps': None, 'metric': 're', 'ode_atol': 1e-07, 'ode_method': 'dopri5', 'ode_rtol': 1e-05, 'ps2_atol': 1e-07, 'ps2_ratio': 2.0, 'ps_method': 'vmf', 'rank': 3, 'renormalize': False, 'save_checkpoint_to_file': False, 'start_time': 0.0, 'step_time': 1, 'stepwise_method': 'mix', 'use_dvr': False, 'visualize_frame': False, 'vmf_atol': 1e-07, 'vmf_reg_method': 'extend', 'vmf_reg_type': 'ip'}
For k=0: s:0.01427897 | e:0.00000000 | a:-0.00371619 | f:0.11949464 | f^2:0.01427897
For k=1: s:0.00592985 | e:0.00354970 | a:0.00417928 | f:-0.07700551 | f^2:0.00592985
For k=2: s:0.00592985 | e:-0.00354970 | a:-0.00417928 | f:0.07700551 | f^2:0.00592985
For k=3: s:0.00322995 | e:0.00178690 | a:0.00054813 | f:-0.05683263 | f^2:0.00322995
For k=4: s:0.0

@3.00 fs:   0%|          | 4/1000 [00:35<3:18:16, 11.94s/it]

Switched to vmf.


@4.00 fs:   0%|          | 5/1000 [02:50<9:26:18, 34.15s/it] 


KeyboardInterrupt: 

# Plotting the Results